In [101]:
from itertools import combinations

 ## Sudoku generator and solver - version 1 ##

In [170]:
sdk1 = '084520000000007000306908000200600800065000740008005002000106508000200000000089230'
sdk = '005006080200030500080005001603010000020604010000070609400800020007020006030900400'
def string_to_board(s):
    board = []
    for i in range(0,81,9):
        row = [int(x) for x in s[i:i+9]]
        board.append(row)
    return board


board = string_to_board(sdk)


In [171]:
def draw_board(board):
    for r in range(9):
        if r % 3 == 0 and r != 0:
            print("-"*21)

        row = ""
        for c in range(9):
            if c % 3 == 0 and c != 0:
                row += "| "

            val = board[r][c] if board[r][c] != 0 else "."
            row += str(val) + " "

        print(row)
draw_board(board)

. . 5 | . . 6 | . 8 . 
2 . . | . 3 . | 5 . . 
. 8 . | . . 5 | . . 1 
---------------------
6 . 3 | . 1 . | . . . 
. 2 . | 6 . 4 | . 1 . 
. . . | . 7 . | 6 . 9 
---------------------
4 . . | 8 . . | . 2 . 
. . 7 | . 2 . | . . 6 
. 3 . | 9 . . | 4 . . 


In [172]:
def get_candidates(board, row, col):
    if board[row][col] != 0:
        return set()

    digits = set(range(1, 10))

    row_nums = set(board[row])

    col_nums = {board[r][col] for r in range(9)}

    start_row = (row // 3) * 3
    start_col = (col // 3) * 3

    box_nums = {
        board[r][c]
        for r in range(start_row, start_row + 3)
        for c in range(start_col, start_col + 3)
    }

    used = row_nums | col_nums | box_nums
    used.discard(0)

    return digits - used

In [173]:
def get_all_candidates(board):
    candidates = [[set() for _ in range(9)] for _ in range(9)]

    for row in range(9):
        for col in range(9):
            if board[row][col] == 0:
                candidates[row][col] = get_candidates(board, row, col)

    return candidates


In [174]:
# Naked Singles

def naked_singles(board, candidates):
    progress = False

    for row in range(9):
        for col in range(9):
            if board[row][col] == 0 and len(candidates[row][col]) == 1:
                value = next(iter(candidates[row][col]))
                board[row][col] = value
                print(f'Populated {value} at {row},{col}')
                progress = True

    return progress

In [175]:
# Hidden Singles

def hidden_singles_rows(board, candidates):

    for row in range(9):
        digit_positions = {digit: [] for digit in range(1, 10)}

        for col in range(9):
            if board[row][col] == 0:
                for digit in candidates[row][col]:
                    digit_positions[digit].append((row, col))

        for digit in range(1,10):
            if len(digit_positions[digit]) == 1:
                r, c = digit_positions[digit][0]
                board[r][c] = digit
                print(f'Hidden ROW single {digit} placed at position {r, c}')
                return True
    return False

def hidden_singles_columns(board, candidates):

    for col in range(9):
        digit_positions = {digit: [] for digit in range(1, 10)}

        for row in range(9):
            if board[row][col] == 0:
                for digit in candidates[row][col]:
                    digit_positions[digit].append((row, col))

        for digit in range(1,10):
            if len(digit_positions[digit]) == 1:
                r, c = digit_positions[digit][0]
                board[r][c] = digit
                print(f'Hidden COLUMN single {digit} placed at position {r, c}')
                return True
    return False

def hidden_singles_boxes(board, candidates):

    for box_row_start in range(0,9,3):
        for box_col_start in range(0,9,3):
            digit_positions = {digit: [] for digit in range(1, 10)}

            for row in range(box_row_start,box_row_start+3):
                for col in range(box_col_start,box_col_start+3):
                    if board[row][col] == 0:
                        for digit in candidates[row][col]:
                            digit_positions[digit].append((row, col))

            for digit in range(1,10):
                if len(digit_positions[digit]) == 1:
                    r, c = digit_positions[digit][0]
                    board[r][c] = digit
                    print(f'Hidden BOX single {digit} placed at position {r, c}')
                    return True
    return False

In [176]:
# Helper Functions

def get_row_cells(row):
    return [(row, col) for col in range(9)]

def get_col_cells(col):
    return [(row, col) for row in range(9)]

def get_box_cells(start_row, start_col):
    return [
        (row, col)
        for row in range(start_row, start_row + 3)
        for col in range(start_col, start_col + 3)
    ]

In [177]:
# Naked Pairs

def naked_sets_in_unit(board, candidates, unit_cells, set_size):
    progress = False

    candidate_cells = [
        (row, col)
        for row, col in unit_cells
        if board[row][col] == 0 and 2 <= len(candidates[row][col]) <= set_size
    ]

    for cell_combo in combinations(candidate_cells, set_size):
        combo_digits = set()

        for row, col in cell_combo:
            combo_digits.update(candidates[row][col])

        if len(combo_digits) == set_size:
            combo_cells = set(cell_combo)

            for row, col in unit_cells:
                if (row, col) not in combo_cells and board[row][col] == 0:
                    before = candidates[row][col].copy()
                    candidates[row][col] -= combo_digits

                    if candidates[row][col] != before:
                        print(
                            f'Naked set {combo_digits} removed from {(row, col)} '
                            f'using cells {cell_combo}'
                        )
                        progress = True

            if progress:
                return True

    return False

def naked_pairs_rows(board, candidates):
    progress = False

    for row in range(9):
        if naked_sets_in_unit(board, candidates, get_row_cells(row), 2):
            progress = True

    return progress

def naked_pairs_cols(board, candidates):
    progress = False

    for col in range(9):
        if naked_sets_in_unit(board, candidates, get_col_cells(col), 2):
            progress = True

    return progress

def naked_pairs_boxes(board, candidates):
    progress = False

    for start_row in range(0, 9, 3):
        for start_col in range(0, 9, 3):
            if naked_sets_in_unit(board, candidates, get_box_cells(start_row, start_col), 2):
                progress = True

    return progress

def naked_pairs(board, candidates):
    if naked_pairs_rows(board, candidates):
        return True
    if naked_pairs_cols(board, candidates):
        return True
    if naked_pairs_boxes(board, candidates):
        return True
    return False

def naked_triples_rows(board, candidates):
    progress = False

    for row in range(9):
        if naked_sets_in_unit(board, candidates, get_row_cells(row), 3):
            progress = True

    return progress

def naked_triples_cols(board, candidates):
    progress = False

    for col in range(9):
        if naked_sets_in_unit(board, candidates, get_col_cells(col), 3):
            progress = True

    return progress

def naked_triples_boxes(board, candidates):
    progress = False

    for start_row in range(0, 9, 3):
        for start_col in range(0, 9, 3):
            if naked_sets_in_unit(board, candidates, get_box_cells(start_row, start_col), 3):
                progress = True

    return progress

def naked_triples(board, candidates):
    if naked_triples_rows(board, candidates):
        return True
    if naked_triples_cols(board, candidates):
        return True
    if naked_triples_boxes(board, candidates):
        return True
    return False

In [178]:
# Hidden Pairs


def hidden_sets_in_unit(board, candidates, unit_cells, set_size):
    progress = False
    digit_positions = {digit: [] for digit in range(1,10)}

    for row, col in unit_cells:
        if board[row][col] == 0:
            for digit in candidates[row][col]:
                digit_positions[digit].append((row, col))

    candidate_digits = [
        digit for digit in range(1, 10)
        if 1 <= len(digit_positions[digit]) <= set_size
    ]

    for digit_combo in combinations(candidate_digits, set_size):
        combo_set = set(digit_combo)
        combo_cells = set()

        for digit in digit_combo:
            combo_cells.update(digit_positions[digit])

        if len(combo_cells) == set_size:
            for row, col in combo_cells:
                before = candidates[row][col].copy()
                candidates[row][col] &= combo_set

                if candidates[row][col] != before:
                    print(
                        f'Hidden set {combo_set} reduced {(row, col)} '
                        f'from {before} to {candidates[row][col]}'
                    )
                    progress = True

            if progress:
                return True

    return False

def hidden_pairs_rows(board, candidates):
    progress = False

    for row in range(9):
        if hidden_sets_in_unit(board, candidates, get_row_cells(row), 2):
            progress = True

    return progress

def hidden_pairs_cols(board, candidates):
    progress = False

    for col in range(9):
        if hidden_sets_in_unit(board, candidates, get_col_cells(col), 2):
            progress = True

    return progress

def hidden_pairs_boxes(board, candidates):
    progress = False

    for start_row in range(0, 9, 3):
        for start_col in range(0, 9, 3):
            if hidden_sets_in_unit(board, candidates, get_box_cells(start_row, start_col), 2):
                progress = True

    return progress

def hidden_pairs(board, candidates):
    if hidden_pairs_rows(board, candidates):
        return True
    if hidden_pairs_cols(board, candidates):
        return True
    if hidden_pairs_boxes(board, candidates):
        return True
    return False

def hidden_triples_rows(board, candidates):
    progress = False

    for row in range(9):
        if hidden_sets_in_unit(board, candidates, get_row_cells(row), 3):
            progress = True

    return progress

def hidden_triples_cols(board, candidates):
    progress = False

    for col in range(9):
        if hidden_sets_in_unit(board, candidates, get_col_cells(col), 3):
            progress = True

    return progress

def hidden_triples_boxes(board, candidates):
    progress = False

    for start_row in range(0, 9, 3):
        for start_col in range(0, 9, 3):
            if hidden_sets_in_unit(board, candidates, get_box_cells(start_row, start_col), 3):
                progress = True

    return progress

def hidden_triples(board, candidates):
    if hidden_triples_rows(board, candidates):
        return True
    if hidden_triples_cols(board, candidates):
        return True
    if hidden_triples_boxes(board, candidates):
        return True

    return False

In [179]:
# Pointing Pairs/Triples

def pointing_pairs_triples(board, candidates):

    for r in range(0,9,3):
        for c in range(0,9,3):
            digit_positions = {digit: [] for digit in range(1,10)}

            for row in range(r, r+3):
                for col in range(c, c+3):
                    if board[row][col] == 0:
                        for digit in candidates[row][col]:
                            digit_positions[digit].append((row,col))

            for digit in range(1,10):
                cells = digit_positions[digit]

                # Rows
                if len(cells) == 2 and cells[0][0] == cells[1][0]:
                    target_row = cells[0][0]
                    print(f'Pointing pair of {digit}s in ROW {target_row}')
                    for x in range(9):
                        if not (c <= x < c+3) and board[target_row][x] == 0:
                            before = candidates[target_row][x].copy()
                            candidates[target_row][x].discard(digit)

                            if candidates[target_row][x] != before:
                                return True

                if len(cells) == 3 and cells[0][0] == cells[1][0] == cells[2][0]:
                    target_row = cells[0][0]
                    print(f'Pointing triple of {digit}s in ROW {target_row}')
                    for x in range(9):
                        if not (c <= x < c+3) and board[target_row][x] == 0:
                            before = candidates[target_row][x].copy()
                            candidates[target_row][x].discard(digit)

                            if candidates[target_row][x] != before:
                                return True

                # Columns
                if len(cells) == 2 and cells[0][1] == cells[1][1]:
                    target_col = cells[0][1]
                    print(f'Pointing pair of {digit}s in COL {target_col}')
                    for x in range(9):
                        if not (r <= x < r+3) and board[x][target_col] == 0:
                            before = candidates[x][target_col].copy()
                            candidates[x][target_col].discard(digit)

                            if candidates[x][target_col] != before:
                                return True

                if len(cells) == 3 and cells[0][1] == cells[1][1] == cells[2][1]:
                    target_col = cells[0][1]
                    print(f'Pointing triple of {digit}s in COL {target_col}')
                    for x in range(9):
                        if not (r <= x < r+3) and board[x][target_col] == 0:
                            before = candidates[x][target_col].copy()
                            candidates[x][target_col].discard(digit)

                            if candidates[x][target_col] != before:
                                return True

    return False

draw_board(board)

. . 5 | . . 6 | . 8 . 
2 . . | . 3 . | 5 . . 
. 8 . | . . 5 | . . 1 
---------------------
6 . 3 | . 1 . | . . . 
. 2 . | 6 . 4 | . 1 . 
. . . | . 7 . | 6 . 9 
---------------------
4 . . | 8 . . | . 2 . 
. . 7 | . 2 . | . . 6 
. 3 . | 9 . . | 4 . . 


In [180]:
# Box-Line Reduction

def box_line_reduction_rows(board, candidates):
    for r in range(9):
        digit_positions = {digit: [] for digit in range(1,10)}
        for c in range(9):
            if board[r][c] == 0:
                for digit in candidates[r][c]:
                    digit_positions[digit].append((r,c))

        for digit in range (1,10):
            cells = digit_positions[digit]

            if len(cells) in {2,3}:
                possible_cell_cols = {cells[x][1] for x in range(len(cells))}

                if possible_cell_cols.issubset({0,1,2}) or possible_cell_cols.issubset({3,4,5}) or possible_cell_cols.issubset({6,7,8}):
                    block_row_start = (r // 3) * 3
                    block_col_start = (cells[0][1] // 3) * 3
                    for row in range(block_row_start, block_row_start + 3):
                        for col in range(block_col_start, block_col_start + 3):
                            if row != r and board[row][col] == 0:
                                before = candidates[row][col].copy()
                                candidates[row][col].discard(digit)

                                if candidates[row][col] != before:
                                    print(f'Found box/line reduction for {digit}s in ROW {r}, removed from BOX {block_row_start},{block_col_start} candidates')
                                    return True
    return False

def box_line_reduction_cols(board, candidates):
    for c in range(9):
        digit_positions = {digit: [] for digit in range(1,10)}
        for r in range(9):
            if board[r][c] == 0:
                for digit in candidates[r][c]:
                    digit_positions[digit].append((r,c))

        for digit in range (1,10):
            cells = digit_positions[digit]

            if len(cells) in {2,3}:
                possible_cell_rows = {cells[x][0] for x in range(len(cells))}

                if possible_cell_rows.issubset({0,1,2}) or possible_cell_rows.issubset({3,4,5}) or possible_cell_rows.issubset({6,7,8}):
                    block_row_start = (cells[0][0] // 3) * 3
                    block_col_start = (c // 3) * 3
                    for row in range(block_row_start, block_row_start + 3):
                        for col in range(block_col_start, block_col_start + 3):
                            if col != c and board[row][col] == 0:
                                before = candidates[row][col].copy()
                                candidates[row][col].discard(digit)

                                if candidates[row][col] != before:
                                    print(f'Found box/line reduction for {digit}s in COL {c}, removed from BOX {block_row_start},{block_col_start} candidates')
                                    return True
    return False

def box_line_reduction(board, candidates):
    if box_line_reduction_rows(board, candidates):
        return True

    if box_line_reduction_cols(board, candidates):
        return True

    return False

draw_board(board)

. . 5 | . . 6 | . 8 . 
2 . . | . 3 . | 5 . . 
. 8 . | . . 5 | . . 1 
---------------------
6 . 3 | . 1 . | . . . 
. 2 . | 6 . 4 | . 1 . 
. . . | . 7 . | 6 . 9 
---------------------
4 . . | 8 . . | . 2 . 
. . 7 | . 2 . | . . 6 
. 3 . | 9 . . | 4 . . 


In [181]:
# X-Wing

def x_wing_rows(board, candidates):
    for digit in range(1, 10):
        row_positions = {}              # key is row number, value is (only) a pair of columns for digit

        for row in range(9):
            cols = []
            for col in range(9):
                if board[row][col] == 0 and digit in candidates[row][col]:
                    cols.append(col)

            if len(cols) == 2:
                row_positions[row] = cols

        rows = list(row_positions.keys())

        for i in range(len(rows)):
            for j in range(i + 1, len(rows)):
                r1, r2 = rows[i], rows[j]

                if row_positions[r1] == row_positions[r2]:
                    c1, c2 = row_positions[r1]

                    for row in range(9):
                        if row not in {r1, r2}:
                            for col in [c1, c2]:
                                if board[row][col] == 0:
                                    before = candidates[row][col].copy()
                                    candidates[row][col].discard(digit)

                                    if candidates[row][col] != before:
                                        print(f'X-Wing: removed {digit} from ({row}, {col}) using rows {r1},{r2} and cols {c1},{c2}')
                                        return True

    return False

def x_wing_cols(board, candidates):
    for digit in range(1, 10):
        col_positions = {}

        for col in range(9):
            rows = []
            for row in range(9):
                if board[row][col] == 0 and digit in candidates[row][col]:
                    rows.append(row)

            if len(rows) == 2:
                col_positions[col] = rows

        cols = list(col_positions.keys())

        for i in range(len(cols)):
            for j in range(i + 1, len(cols)):
                c1, c2 = cols[i], cols[j]

                if col_positions[c1] == col_positions[c2]:
                    r1, r2 = col_positions[c1]

                    for col in range(9):
                        if col not in {c1, c2}:
                            for row in [r1, r2]:
                                if board[row][col] == 0:
                                    before = candidates[row][col].copy()
                                    candidates[row][col].discard(digit)

                                    if candidates[row][col] != before:
                                        print(f'X-Wing: removed {digit} from ({row}, {col}) using rows {r1},{r2} and cols {c1},{c2}')
                                        return True

    return False

def x_wing(board, candidates):
    if x_wing_rows(board, candidates):
        return True
    if x_wing_cols(board, candidates):
        return True
    return False


In [182]:
# Y-Wing

def cells_see_each_other(cell1, cell2):
    r1, c1 = cell1
    r2, c2 = cell2

    return (
        r1 == r2 or
        c1 == c2 or
        (r1 // 3 == r2 // 3 and c1 // 3 == c2 // 3)
    )

def get_peers(row, col):
    peers = set()

    for c in range(9):
        if c != col:
            peers.add((row, c))

    for r in range(9):
        if r != row:
            peers.add((r, col))

    box_row_start = (row // 3) * 3
    box_col_start = (col // 3) * 3

    for r in range(box_row_start, box_row_start + 3):
        for c in range(box_col_start, box_col_start + 3):
            if (r, c) != (row, col):
                peers.add((r, c))

    return peers

def y_wing(board, candidates):
    bivalue_cells = []

    # collect all cells with exactly 2 candidates
    for row in range(9):
        for col in range(9):
            if board[row][col] == 0 and len(candidates[row][col]) == 2:
                bivalue_cells.append((row, col))

    # try each bivalue cell as the pivot
    for pivot in bivalue_cells:
        pr, pc = pivot
        pivot_set = candidates[pr][pc]

        # find first wing
        for wing1 in bivalue_cells:
            if wing1 == pivot:
                continue

            if not cells_see_each_other(pivot, wing1):
                continue

            w1r, w1c = wing1
            wing1_set = candidates[w1r][w1c]

            shared1 = pivot_set & wing1_set
            if len(shared1) != 1:
                continue

            # find second wing
            for wing2 in bivalue_cells:
                if wing2 == pivot or wing2 == wing1:
                    continue

                if not cells_see_each_other(pivot, wing2):
                    continue

                w2r, w2c = wing2
                wing2_set = candidates[w2r][w2c]

                shared2 = pivot_set & wing2_set
                if len(shared2) != 1:
                    continue

                # wings must share different pivot digits
                if shared1 == shared2:
                    continue

                # together, pivot + wings must use exactly 3 digits
                union_digits = pivot_set | wing1_set | wing2_set
                if len(union_digits) != 3:
                    continue

                # the digit shared by the two wings but not in the pivot is the elimination digit
                elim_digit_set = (wing1_set & wing2_set) - pivot_set
                if len(elim_digit_set) != 1:
                    continue

                elim_digit = next(iter(elim_digit_set))

                # eliminate from cells that see both wings
                common_peers = get_peers(w1r, w1c) & get_peers(w2r, w2c)

                for row, col in common_peers:
                    if board[row][col] == 0:
                        before = candidates[row][col].copy()
                        candidates[row][col].discard(elim_digit)

                        if candidates[row][col] != before:
                            print(
                                f'Y-Wing: removed {elim_digit} from ({row}, {col}) '
                                f'using pivot {pivot}, wing1 {wing1}, wing2 {wing2}'
                            )
                            return True

    return False


In [183]:
# Swordfish

def swordfish_rows(board, candidates):
    for digit in range(1, 10):
        row_positions = {}

        for row in range(9):
            cols = []
            for col in range(9):
                if board[row][col] == 0 and digit in candidates[row][col]:
                    cols.append(col)

            if 2 <= len(cols) <= 3:
                row_positions[row] = set(cols)

        rows = list(row_positions.keys())

        for row_combo in combinations(rows, 3):
            union_cols = set()
            for row in row_combo:
                union_cols.update(row_positions[row])

            if len(union_cols) == 3:
                for row in range(9):
                    if row not in row_combo:
                        for col in union_cols:
                            if board[row][col] == 0:
                                before = candidates[row][col].copy()
                                candidates[row][col].discard(digit)

                                if candidates[row][col] != before:
                                    print(
                                        f'Swordfish: removed {digit} from ({row}, {col}) '
                                        f'using rows {row_combo} and cols {sorted(union_cols)}'
                                    )
                                    return True
    return False

def swordfish_cols(board, candidates):
    for digit in range(1, 10):
        col_positions = {}

        for col in range(9):
            rows = []
            for row in range(9):
                if board[row][col] == 0 and digit in candidates[row][col]:
                    rows.append(row)

            if 2 <= len(rows) <= 3:
                col_positions[col] = set(rows)

        cols = list(col_positions.keys())

        for col_combo in combinations(cols, 3):
            union_rows = set()
            for col in col_combo:
                union_rows.update(col_positions[col])

            if len(union_rows) == 3:
                for col in range(9):
                    if col not in col_combo:
                        for row in union_rows:
                            if board[row][col] == 0:
                                before = candidates[row][col].copy()
                                candidates[row][col].discard(digit)

                                if candidates[row][col] != before:
                                    print(
                                        f'Swordfish: removed {digit} from ({row}, {col}) '
                                        f'using cols {col_combo} and rows {sorted(union_rows)}'
                                    )
                                    return True
    return False

def swordfish(board, candidates):
    if swordfish_rows(board, candidates):
        return True
    if swordfish_cols(board, candidates):
        return True
    return False

In [184]:
# Skyscraper

def skyscraper_rows(board, candidates):
    for digit in range(1, 10):
        row_positions = {}

        for row in range(9):
            cols = []
            for col in range(9):
                if board[row][col] == 0 and digit in candidates[row][col]:
                    cols.append(col)

            if len(cols) == 2:
                row_positions[row] = set(cols)

        rows = list(row_positions.keys())

        for i in range(len(rows)):
            for j in range(i + 1, len(rows)):
                r1, r2 = rows[i], rows[j]

                common_cols = row_positions[r1] & row_positions[r2]

                if len(common_cols) == 1:
                    shared_col = next(iter(common_cols))
                    roof1_col = next(iter(row_positions[r1] - {shared_col}))
                    roof2_col = next(iter(row_positions[r2] - {shared_col}))

                    roof1 = (r1, roof1_col)
                    roof2 = (r2, roof2_col)

                    common_peers = get_peers(*roof1) & get_peers(*roof2)

                    for row, col in common_peers:
                        if board[row][col] == 0:
                            before = candidates[row][col].copy()
                            candidates[row][col].discard(digit)

                            if candidates[row][col] != before:
                                print(
                                    f'Skyscraper: removed {digit} from ({row}, {col}) '
                                    f'using rows {r1},{r2} with shared col {shared_col} '
                                    f'and roofs {roof1}, {roof2}'
                                )
                                return True
    return False

def skyscraper_cols(board, candidates):
    for digit in range(1, 10):
        col_positions = {}

        for col in range(9):
            rows = []
            for row in range(9):
                if board[row][col] == 0 and digit in candidates[row][col]:
                    rows.append(row)

            if len(rows) == 2:
                col_positions[col] = set(rows)

        cols = list(col_positions.keys())

        for i in range(len(cols)):
            for j in range(i + 1, len(cols)):
                c1, c2 = cols[i], cols[j]

                common_rows = col_positions[c1] & col_positions[c2]

                if len(common_rows) == 1:
                    shared_row = next(iter(common_rows))
                    roof1_row = next(iter(col_positions[c1] - {shared_row}))
                    roof2_row = next(iter(col_positions[c2] - {shared_row}))

                    roof1 = (roof1_row, c1)
                    roof2 = (roof2_row, c2)

                    common_peers = get_peers(*roof1) & get_peers(*roof2)

                    for row, col in common_peers:
                        if board[row][col] == 0:
                            before = candidates[row][col].copy()
                            candidates[row][col].discard(digit)

                            if candidates[row][col] != before:
                                print(
                                    f'Skyscraper: removed {digit} from ({row}, {col}) '
                                    f'using cols {c1},{c2} with shared row {shared_row} '
                                    f'and roofs {roof1}, {roof2}'
                                )
                                return True
    return False

def skyscraper(board, candidates):
    if skyscraper_rows(board, candidates):
        return True
    if skyscraper_cols(board, candidates):
        return True
    return False

In [185]:
# Validation checks
def is_valid_row(board, row):
    desired_set = {1,2,3,4,5,6,7,8,9}
    digits = set()

    for cell in range(9):
        digits.add(board[row][cell])

    if digits == desired_set:
        return True
    else:
        return False

def is_valid_col(board, col):
    desired_set = {1,2,3,4,5,6,7,8,9}
    digits = set()
    for cell in range(9):
        digits.add(board[cell][col])

    if digits == desired_set:
        return True
    else:
        return False

def is_valid_block(board, row, col):
    desired_set = {1,2,3,4,5,6,7,8,9}
    digits = set()

    for r in range(row, row+3):
        for c in range(col, col+3):
            digits.add(board[r][c])

    if digits == desired_set:
        return True
    else:
        return False

def is_solved(board):
    errors = 0
    for row in range(9):
        is_valid = is_valid_row(board, row)
        if not is_valid:
            errors += 1
            print(f'Board not valid at row {row+1}')

    for col in range(9):
        is_valid = is_valid_col(board, col)
        if not is_valid:
            errors += 1
            print(f'Board not valid at col {col+1}')

    for row in range(0,9,3):
        for col in range(0,9,3):
            is_valid = is_valid_block(board, row, col)
            if not is_valid:
                errors += 1
                print(f'Board not valid at block ({row+1}, {col+1})')

    if errors == 0:
        print(f'Board is solved, great success')
    else:
        print(f'Board is obviously not solved :(')


In [186]:
def run_solver(board):
    candidates = get_all_candidates(board)
    while True:

        if naked_singles(board, candidates):
            candidates = get_all_candidates(board)
            continue

        if hidden_singles_rows(board, candidates):
            candidates = get_all_candidates(board)
            continue

        if hidden_singles_columns(board, candidates):
            candidates = get_all_candidates(board)
            continue

        if hidden_singles_boxes(board, candidates):
            candidates = get_all_candidates(board)
            continue

        if naked_pairs(board, candidates):
            continue

        if naked_triples(board, candidates):
            continue

        if hidden_pairs(board, candidates):
            continue

        if hidden_triples(board, candidates):
            continue

        if pointing_pairs_triples(board, candidates):
            continue

        if box_line_reduction(board, candidates):
            continue

        if x_wing(board, candidates):
            continue

        if y_wing(board, candidates):
            continue

        if swordfish(board, candidates):
            continue

        if skyscraper(board, candidates):
            continue

        print("No more progress can be made.")
        draw_board(board)
        break

run_solver(board)


Hidden ROW single 8 placed at position (1, 5)
Hidden ROW single 4 placed at position (7, 3)
Hidden ROW single 2 placed at position (8, 2)
Hidden ROW single 6 placed at position (8, 4)
Populated 5 at 6,4
Hidden COLUMN single 3 placed at position (5, 3)
Populated 2 at 5,5
Populated 5 at 3,3
Populated 9 at 3,5
Populated 8 at 4,4
Populated 9 at 4,2
Hidden COLUMN single 8 placed at position (5, 2)
Naked set {4, 7} removed from (3, 6) using cells ((3, 1), (3, 7))
Naked set {4, 7} removed from (3, 8) using cells ((3, 1), (3, 7))
Naked set {4, 5, 7} removed from (1, 7) using cells ((3, 7), (5, 7), (8, 7))
Naked set {4, 5, 7} removed from (2, 7) using cells ((3, 7), (5, 7), (8, 7))
Naked set {4, 5, 7} removed from (7, 7) using cells ((3, 7), (5, 7), (8, 7))
Pointing pair of 3s in COL 0
Pointing pair of 1s in COL 3
Pointing pair of 2s in COL 3
Pointing pair of 4s in COL 4
Pointing triple of 7s in COL 3
Pointing pair of 9s in COL 4
Pointing pair of 4s in COL 8
Pointing pair of 6s in COL 7
Pointin

Board is solved, great success
